In [ ]:
import pandas as pd
import numpy as np

activities = pd.read_csv(
    "../data/raw/chembl_leishmania_donovani_activities.csv",
    sep=";",
    engine="python",
    on_bad_lines="skip"
)

print(activities.shape)

This created activities_nm which contains compounds having their Standard Units as nM only.

In [ ]:
activities_nm=activities[activities["Standard Units"]=="nM"].copy()

print(activities_nm.shape)

We checked for how many compounds have null Standard value

In [ ]:
activities_nm["Standard Value"].isnull().sum()

The compounds having null Standard values were deleted.

In [ ]:
activities_nm=activities_nm.dropna(subset=["Standard Value"])
print(activities_nm.shape)

It was verified if the compounds having null Standard value have been deleted.

In [ ]:
activities_nm["Standard Value"].isnull().sum()

The pIC50 values were calculated using the formula:
pIC50 = 9 - log10(IC50 [nM])

This converts the IC50 value in nanomolar units into a logarithmic potency score.

In [ ]:
activities_nm["pIC50"] = (
    9 - np.log10(activities_nm["Standard Value"])
)
activities_nm["pIC50"].describe()

## Checking for Invalid Activity Values

The dataset was examined for compounds with IC50 values less than or equal to zero. Such values are invalid for pIC50 calculations because logarithmic transformations cannot be applied to zero or negative values.

In [ ]:
activities_nm[
    activities_nm["Standard Value"] <= 0
][
    [
        "Molecule ChEMBL ID",
        "Standard Value"
    ]
]

Number of compounds with  IC50 values less than or equal to zero.

In [ ]:
(activities_nm["Standard Value"] <= 0).sum()

## Removal of Invalid Records and Recalculation of pIC50

Records with IC50 values less than or equal to zero were removed from the dataset. pIC50 values were then recalculated, and descriptive statistics were generated to verify the distribution of the cleaned activity data.

In [ ]:
activities_nm = activities_nm[
    activities_nm["Standard Value"] > 0
].copy()
activities_nm["pIC50"] = (
    9 - np.log10(activities_nm["Standard Value"])
)
activities_nm["pIC50"].describe()

## Identification of Extreme Activity Outliers

The dataset was sorted by pIC50 values to identify compounds with unusually low activity. This step was performed to detect potential outliers, data entry errors, or biologically insignificant compounds that could negatively impact downstream analyses and machine learning model performance.

In [ ]:
activities_nm.sort_values(
    "pIC50"
).head(20)[
    [
        "Molecule ChEMBL ID",
        "Standard Value",
        "pIC50"
    ]
]


## Summary Statistics of IC50 Values

Descriptive statistics of the IC50 values were generated to examine the overall distribution, central tendency, and range of activity measurements. This helped identify potential outliers and assess the quality of the bioactivity data before further cleaning and analysis.

In [ ]:
activities_nm["Standard Value"].describe()

## Quantifying Extreme Outliers

The number of compounds with pIC50 values below 2 was determined. These compounds represent extremely weak activity and were assessed as potential outliers prior to constructing the final cleaned dataset.

In [ ]:
(activities_nm["pIC50"] < 2).sum()

## Removal of Extreme Activity Outliers

Compounds with pIC50 values below 2 were removed from the dataset. These records correspond to extremely weak activity and were considered outliers. The remaining dataset was retained for subsequent analysis and machine learning model development.

In [ ]:
activities_nm = activities_nm[
    activities_nm["pIC50"] >= 2
].copy()

print(activities_nm.shape)

## Investigation of Duplicate Compounds

Multiple activity measurements may exist for the same compound due to testing in different assays or experimental studies. Duplicate compound records were examined prior to creating the final compound-level dataset and number of duplicates were obtained.

In [ ]:
activities_nm["Molecule ChEMBL ID"].duplicated().sum()

## Handling Duplicate Compounds

Duplicate records were grouped by Molecule ChEMBL ID to create a single entry per compound. The median pIC50 value was retained as the representative activity because it is less affected by outliers than the mean. The corresponding SMILES structure was also preserved for each compound.

In [ ]:
activities_clean = (
    activities_nm
    .groupby("Molecule ChEMBL ID")
    .agg({
        "pIC50": "median",
        "Smiles": "first"
    })
    .reset_index()
)

## Final Cleaned Activity Dataset

After removing invalid records, filtering extreme outliers, and consolidating duplicate compounds, the final activity dataset contained 6,332 unique compounds. Each record consists of a Molecule ChEMBL ID, a corresponding SMILES structure, and a representative pIC50 value.

In [ ]:
print(activities_clean.shape)

The initial 5 rows and all 3 columns of the new dataset.

In [ ]:
activities_clean.head()

The cleaned dataset was saved to the folder 'data_clean' as 'activities_clean.csv'

In [ ]:
activities_clean.to_csv(
    "../data/processed/activities_clean.csv",
    index=False
)

In [ ]:
activities_clean.columns